In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

In [2]:
np.random.seed(42)

### PARAMETERS

In [3]:
NUM_PATIENTS = 1000
NUM_VISITS = 5000
NUM_FACILITIES = 40

states = ["Lagos", "Kano", "Kaduna", "Enugu", "Plateau", "Oyo", "Rivers", "Borno"]
facility_types = ["Primary", "Secondary", "Tertiary"]
ownership_types = ["Public", "Private"]
urban_rural = ["Urban", "Rural"]

diseases = [
    ("Malaria", "Communicable", 0),
    ("Tuberculosis", "Communicable", 0),
    ("Measles", "Communicable", 1),
    ("COVID-19", "Communicable", 1),
    ("Cholera", "Communicable", 0),
    ("Hypertension", "NCD", 0),
    ("Diabetes", "NCD", 0),
    ("Asthma", "NCD", 0),
    ("Stroke", "NCD", 0),
    ("Chronic Kidney Disease", "NCD", 0)
]

### PATIENTS

In [6]:
start_date = pd.to_datetime('1950-01-01')
end_date = pd.to_datetime('2022-01-01')

days_between = (end_date - start_date).days

patients = pd.DataFrame({
    "patient_id": range(1, NUM_PATIENTS + 1),
    "gender": np.random.choice(["Male", "Female"], NUM_PATIENTS),
    "birth_date": [start_date + pd.Timedelta(days=np.random.randint(0, days_between)) 
                   for _ in range(NUM_PATIENTS)],
    "state": np.random.choice(states, NUM_PATIENTS)
})

patients['birth_date'] = patients['birth_date'].dt.date

patients.to_csv("../03_csv_data/patients.csv", index=False)

### FACILITIES

In [7]:
facilities = pd.DataFrame({
    "facility_id": range(1, NUM_FACILITIES + 1),
    "facility_name": [f"Facility_{i}" for i in range(1, NUM_FACILITIES + 1)],
    "state": np.random.choice(states, NUM_FACILITIES),
    "facility_type": np.random.choice(facility_types, NUM_FACILITIES),
    "ownership": np.random.choice(ownership_types, NUM_FACILITIES),
    "urban_rural": np.random.choice(urban_rural, NUM_FACILITIES)
})

facilities.to_csv("../03_csv_data/facilities.csv", index=False)

### DISEASES

In [8]:
disease_df = pd.DataFrame(diseases, columns=["disease_name", "category", "is_vaccine_preventable"])
disease_df["disease_id"] = range(1, len(disease_df)+1)
disease_df = disease_df[["disease_id", "disease_name", "category", "is_vaccine_preventable"]]

disease_df.to_csv("../03_csv_data/diseases.csv", index=False)

### VISITS

In [9]:
start_date = datetime(2020,1,1)
end_date = datetime(2024,12,31)

def random_date():
    return start_date + timedelta(days=random.randint(0, (end_date - start_date).days))

visit_disease_probs = [0.25, 0.07, 0.05, 0.10, 0.05, 0.18, 0.10, 0.08, 0.07, 0.05]

visits = []

for i in range(1, NUM_VISITS + 1):
    disease_id = np.random.choice(range(1,11), p=visit_disease_probs)
    visit_date = random_date()
    severity = np.random.choice(["Mild","Moderate","Severe"], p=[0.6,0.3,0.1])
    
    outcome = "Recovered"
    if severity == "Severe":
        if random.random() < 0.15:
            outcome = "Death"
    
    visits.append([
        i,
        np.random.randint(1, NUM_PATIENTS+1),
        np.random.randint(1, NUM_FACILITIES+1),
        visit_date.date(),
        disease_id,
        severity,
        outcome
    ])

visits_df = pd.DataFrame(visits, columns=[
    "visit_id","patient_id","facility_id","visit_date",
    "disease_id","severity","outcome"
])

visits_df.to_csv("../03_csv_data/visits.csv", index=False)

### VACCINATIONS

In [10]:
vaccines = ["Measles Vaccine", "COVID-19 Vaccine"]
vaccinations = []
vaccination_id = 1

# Ensure datetime
patients["birth_date"] = pd.to_datetime(patients["birth_date"])

# Lookup dictionaries
birth_lookup = dict(zip(patients["patient_id"], patients["birth_date"]))
state_lookup = dict(zip(patients["patient_id"], patients["state"]))

max_vaccine_date = datetime(2025, 12, 31)


# State-level vaccination inequalities
state_coverage = {
    "Lagos": 0.85,
    "Oyo": 0.80,
    "Rivers": 0.78,
    "Kaduna": 0.70,
    "Kano": 0.65,
    "Plateau": 0.72,
    "Enugu": 0.82,
    "Borno": 0.55
}

# Vaccine-specific baseline coverage
vaccine_base_coverage = {
    "Measles Vaccine": 0.85,   # childhood vaccine
    "COVID-19 Vaccine": 0.60   # lower adult uptake
}

for vaccine_name in vaccines:

    base_cov = vaccine_base_coverage[vaccine_name]

    # Iterate per state to introduce inequality
    for state in state_coverage:

        # Get patients in this state
        state_patients = patients.loc[
            patients["state"] == state, "patient_id"
        ].values

        if len(state_patients) == 0:
            continue

        # Final coverage = base vaccine uptake * state access modifier
        final_coverage = base_cov * state_coverage[state]

        n_vaccinated = int(len(state_patients) * final_coverage)

        if n_vaccinated == 0:
            continue

        vaccinated_patients = np.random.choice(
            state_patients,
            size=n_vaccinated,
            replace=False
        )

        # Generate vaccination records
        for patient_id in vaccinated_patients:

            facility_id = np.random.randint(1, NUM_FACILITIES + 1)
            birth_date = birth_lookup[patient_id]

            # Minimum age eligibility (9 months)
            min_vaccine_date = birth_date + timedelta(days=280)

            # Vaccine rollout rules
            if vaccine_name == "COVID-19 Vaccine":
                covid_rollout = datetime(2020, 12, 11)
                min_vaccine_date = max(min_vaccine_date, covid_rollout)

            if vaccine_name == "Measles Vaccine":
                measles_min = datetime(1970, 1, 1)
                min_vaccine_date = max(min_vaccine_date, measles_min)

            # Skip impossible timelines
            if min_vaccine_date > max_vaccine_date:
                continue

            # Random vaccination date within valid window
            max_days = (max_vaccine_date - min_vaccine_date).days
            vaccination_date = min_vaccine_date + timedelta(
                days=np.random.randint(0, max_days + 1)
            )

            # dose logic
            dose_number = 1
            if vaccine_name == "COVID-19 Vaccine":
                dose_number = np.random.choice([1, 2], p=[0.7, 0.3])

            vaccinations.append([
                vaccination_id,
                patient_id,
                vaccine_name,
                dose_number,
                vaccination_date.date(),
                facility_id
            ])
            vaccination_id += 1

# Convert to DataFrame
vaccinations_df = pd.DataFrame(vaccinations, columns=[
    "vaccination_id",
    "patient_id",
    "vaccine_name",
    "dose_number",
    "vaccination_date",
    "facility_id"
])

vaccinations_df.to_csv("../03_csv_data/vaccinations.csv", index=False)

### TREATMENTS

In [11]:
treatment_types = ["Drug", "Surgery", "Therapy", "Dialysis"]

treatments = []

for i, row in visits_df.iterrows():
    visit_id = row["visit_id"]
    severity = row["severity"]
    
    # Base cost logic
    if severity == "Mild":
        cost = np.random.uniform(5000, 20000)
    elif severity == "Moderate":
        cost = np.random.uniform(20000, 80000)
    else:  # Severe
        cost = np.random.uniform(15000, 300000)
    
    treatment_type = np.random.choice(
        treatment_types,
        p=[0.6, 0.1, 0.2, 0.1]  # drugs most common
    )
    
    treatments.append([
        i + 1,
        visit_id,
        treatment_type,
        round(cost, 2)
    ])

treatments_df = pd.DataFrame(treatments, columns=[
    "treatment_id", "visit_id", "treatment_type", "cost"
])

treatments_df.to_csv("../03_csv_data/treatments.csv", index=False)